# Exp 10 - Edge Computing-based Obstacle Detection using Python

> Scope note: this notebook is a lab-scale deterministic simulation. It is intended for understanding timing, communication, and security concepts. It is not a certification model for a production autonomous vehicle.

## Objective

Simulate an edge-side obstacle decision loop for autonomous systems.

The notebook models a lightweight decision rule: if an obstacle is close enough and the confidence is high enough, the edge node triggers a brake action. This is not computer vision; it is a deterministic control-decision simulation.

## Textbook Notes and Case Studies

### 1. Textbook Background

Edge computing moves computation closer to the data source. In autonomous systems, this reduces the delay that would occur if every sensor frame or telemetry event had to travel to a remote cloud service before a decision is made. Obstacle detection is a strong edge-computing use case because delayed detection can make the result useless.

The lab model usually simplifies perception into distance thresholds or object lists. A real obstacle-detection pipeline may include camera, lidar, radar, sensor fusion, neural-network inference, tracking, and decision logic. The important lesson is the timing budget across the whole chain.

### 2. Architecture Notes

```
Sensor Input -> Preprocessing -> Obstacle Detection -> Decision Rule -> Alert/Actuation
      |              |                 |                 |
      v              v                 v                 v
Frame/Data       Filtering        Distance/Confidence  Brake/Warning
```

An edge node should produce timely, local decisions. The cloud may still be used for model training, fleet analytics, logging, or non-urgent optimization.

### 3. Important Formulas

End-to-end detection delay:

```
detection_delay = capture_time + preprocessing_time + inference_time + decision_time
```

Stopping distance approximation:

```
stopping_distance = reaction_distance + braking_distance
```

where:

```
reaction_distance = speed * reaction_time
braking_distance = speed^2 / (2 * deceleration)
```

Alert condition:

```
alert = obstacle_distance <= safety_threshold
```

### 4. Classroom Case Studies

Case Study A - Warehouse Robot:
A mobile robot detects a pallet in its path. Edge detection is preferred because waiting for a cloud round trip would increase stopping distance.

Case Study B - Roadside Edge Camera:
A roadside edge unit detects pedestrians and sends warnings to vehicles. The benefit comes from local processing near the camera rather than remote cloud processing.

Case Study C - Drone Navigation:
A drone uses onboard obstacle detection because wireless connectivity may be unreliable and low-altitude reaction time is short.

### 5. Analysis Checklist

Report obstacle distance, threshold, decision result, and detection delay. If using a simplified model, state that it is threshold-based and not a full computer-vision detector.

### 6. Source Notes

- Edge computing latency motivation is consistent with NIST fog/edge computing discussions: https://www.nist.gov/programs-projects/fog-computing
- Python math support for distance and threshold calculations: https://docs.python.org/3/library/math.html


## Architecture

```text
Sensor / Perception Output
  |-- obstacle distance
  |-- detection confidence
          |
          v
Edge Decision Node
  |-- distance threshold
  |-- confidence threshold
          |
          v
Action
  |-- BRAKE
  |-- MONITOR
          |
          v
Threshold Sensitivity Analysis
```

## Formulas and Required Theory

Decision rule:

\[
\text{BRAKE} =
\begin{cases}
1, & distance < D_{threshold} \land confidence \ge C_{threshold}\\
0, & otherwise
\end{cases}
\]

Detection rate for threshold testing:

\[
\text{detection count} = \sum_{i=1}^{n} \mathbb{1}(distance_i < D_{threshold})
\]

Lower thresholds reduce false braking but can miss obstacles. Higher thresholds detect earlier but can increase unnecessary braking.

## In-Lab Method

1. Generate frame-level obstacle distances and confidence values.
2. Apply a distance threshold.
3. Apply a confidence threshold.
4. Output either `BRAKE` or `MONITOR`.
5. Inspect how decisions change across frames.

In [1]:
import random

print("EXP 10 - IN-LAB EDGE OBSTACLE DETECTION")
rng = random.Random(341410)
threshold_m = 8.0
for frame in range(1, 11):
    distance = round(rng.uniform(3.0, 18.0), 2)
    confidence = round(rng.uniform(0.70, 0.99), 2)
    action = "BRAKE" if distance < threshold_m and confidence >= 0.75 else "MONITOR"
    print(f"frame={frame:02d} distance={distance:5.2f} m confidence={confidence:.2f} action={action}")

EXP 10 - IN-LAB EDGE OBSTACLE DETECTION
frame=01 distance=10.89 m confidence=0.82 action=MONITOR
frame=02 distance=17.93 m confidence=0.81 action=MONITOR
frame=03 distance=13.20 m confidence=0.97 action=MONITOR
frame=04 distance= 4.55 m confidence=0.87 action=BRAKE
frame=05 distance= 4.22 m confidence=0.79 action=BRAKE
frame=06 distance= 9.12 m confidence=0.97 action=MONITOR
frame=07 distance= 6.38 m confidence=0.87 action=BRAKE
frame=08 distance=15.80 m confidence=0.84 action=MONITOR
frame=09 distance=15.45 m confidence=0.97 action=MONITOR
frame=10 distance=16.76 m confidence=0.95 action=MONITOR


## Post-Lab Method

The post-lab cell varies the distance threshold and counts detections over 100 simulated distances. This gives a threshold-sensitivity view.

In [2]:
import random

print("EXP 10 - POST-LAB THRESHOLD SENSITIVITY")
rng = random.Random(341411)
distances = [rng.uniform(2, 20) for _ in range(100)]
for threshold in [5, 8, 12, 15]:
    detections = sum(d < threshold for d in distances)
    print(f"threshold={threshold:2} m detections={detections:3}/100 false-brake-risk={'higher' if threshold >= 12 else 'moderate'}")

EXP 10 - POST-LAB THRESHOLD SENSITIVITY
threshold= 5 m detections= 11/100 false-brake-risk=moderate
threshold= 8 m detections= 30/100 false-brake-risk=moderate
threshold=12 m detections= 58/100 false-brake-risk=higher
threshold=15 m detections= 77/100 false-brake-risk=higher


## What to Write in the Lab Record

- Include several frame decisions.
- State the threshold values used.
- Discuss the trade-off between early detection and false brake risk.
- Mention that production systems require sensor validation, perception testing, and fail-safe design.

## References

- Python `time` module documentation: https://docs.python.org/3/library/time.html
- Python `statistics` module documentation: https://docs.python.org/3/library/statistics.html
- Python `random` module documentation: https://docs.python.org/3/library/random.html